<a href="https://colab.research.google.com/github/knutzin/tcc_clickbait/blob/main/tcc_clickbait.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Testando GPU

In [1]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


Instalando bibliotecas

In [2]:
!pip install -q transformers accelerate scikit-learn pandas numpy fastapi uvicorn

Rodar code Clickbait_bertimbau.py

In [ ]:
!python /content/clickbait_bertimbau.py train --csv "/content/novo dataset + noticias clickbait csv.csv"

Classificado errado - Falso negativo

In [ ]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_dir = "/content/saida_bertimbau/modelo_bertimbau_clickbait"

test_df = pd.read_csv(
    "/content/saida_bertimbau/split_test.csv",
    sep=";"
)

tokenizer = AutoTokenizer.from_pretrained(model_dir)
model = AutoModelForSequenceClassification.from_pretrained(model_dir)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

probabilidades = []

for titulo in test_df["titulo"]:
    inputs = tokenizer(
        titulo,
        return_tensors="pt",
        truncation=True,
        max_length=128
    ).to(device)

    with torch.no_grad():
        logits = model(**inputs).logits
        prob = torch.softmax(logits, dim=1)[0][1].item()
        probabilidades.append(prob)

test_df["prob_clickbait"] = probabilidades

# Use o limiar do novo treinamento
test_df["predicao"] = (
    test_df["prob_clickbait"] >= 0.865
).astype(int)

erros = test_df[
    (test_df["clickbait_label_v2"] == 1) &
    (test_df["predicao"] == 0)
]

print("Quantidade:", len(erros))

resultado = erros[[
    "titulo",
    "clickbait_label_v2",
    "predicao",
    "prob_clickbait"
]]

display(resultado)

# Salvar em Excel
resultado.to_excel(
    "/content/clickbaits_nao_detectados.xlsx",
    index=False
)

print("Arquivo criado: clickbaits_nao_detectados.xlsx")